# Temporal Spectral Embedding on harxhar residuals

This notebook is the **source of truth** for the spectral embedding machinery.
Cells marked `# export` get auto-exported to
`src/features/extractors/spectral_embedding.py` by hpc-agent's auto-export
step; the rest is exploration on real 30-min RV data.

Pipeline mirrors what `src.models.spectral_knn.fit_predict_spectral_knn`
does at refit time:

1. Load + transform: `adj_RV`, HAR lags, calendar features.
2. Whole-series rolling-robust-scale the feature matrix.
3. Fit a single Ridge on a warm-up window; OOS predictions on the rest.
4. Residuals = `adj_RV - ridge.predict(X)`.
5. Slide a `W`-bar window over residuals; subsample to one view per day.
6. Build the embedding (the machinery defined below) and plot.

In [ ]:
from __future__ import annotations

import os
import sys
from pathlib import Path

REPO = Path.cwd()
while REPO.parent != REPO and not (REPO / "src").is_dir():
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
os.chdir(REPO)

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.colors import LogNorm
from sklearn.linear_model import Ridge

from src.backtest.executor import _build_har_and_calendar, load_and_transform
from src.features.transforms.scaling import rolling_robust_scale
from src.features.transforms.target import PERIODS_PER_DAY

## 1. Load + HAR-prep the real data

Same data prep as the spectral_knn pipeline: `load_and_transform` with
diurnal-adjusted RV target, 240-period winsorization, NaN-drop; then `_build_har_and_calendar`
for the 13 features (6 HAR rolling means at base-5 lags + 5 DOW dummies + hour + overnight flag).
Target is `adj_RV` shifted by `horizon` rows.

In [ ]:
HORIZON = 1

df, _ = load_and_transform(
    "data", exog_cols=[],
    target_use_diurnal=True, target_winsor_window=240, dropna_with_exog=True,
)
df, feature_names = _build_har_and_calendar(df, exog_cols=[], add_calendar=True)
df["target"] = df["adj_RV"].shift(-HORIZON)
df = df.dropna(subset=["target"] + feature_names).reset_index(drop=True)
df["t"] = pd.to_datetime(df["t"])

print(f"rows:             {len(df):,}")
print(f"date range:       {df['t'].min()}  ..  {df['t'].max()}")
print(f"feature columns:  {len(feature_names)}  -> {feature_names}")

## 2. Whole-series rolling-robust-scale + Ridge baseline

* Scale: each HAR/calendar column rolling-normalized by its median / IQR over the trailing
  `train_win = train_window * 48` bars. Mirrors the prescaling step `ml_ridge` does.
* Baseline: a single Ridge fit on the first `WARMUP_DAYS` of data. Then OOS predict on
  everything after `WARMUP_DAYS`. Residuals = `target - ridge.predict(X_scaled)` on the
  OOS region.

A real walk-forward would refit Ridge every step; here we freeze one fit so the notebook
stays fast and the visualization isn't muddied by per-step refit drift. The embedding's
qualitative structure is the same.

In [ ]:
WARMUP_DAYS = 500
train_win = WARMUP_DAYS * PERIODS_PER_DAY
RIDGE_ALPHA = 1.0

X = df[feature_names].to_numpy(dtype=np.float64)
y = df["target"].to_numpy(dtype=np.float64)

X_scaled = rolling_robust_scale(X, train_win)

ridge = Ridge(alpha=RIDGE_ALPHA).fit(X_scaled[:train_win], y[:train_win])
y_hat = ridge.predict(X_scaled[train_win:])
residuals = y[train_win:] - y_hat
dates_resid = df["t"].iloc[train_win:].reset_index(drop=True)

print(f"residuals: {residuals.shape}    OOS region: {dates_resid.iloc[0]}  ..  {dates_resid.iloc[-1]}")
print(f"residual std: {residuals.std():.4f}   |    adj_RV std (OOS): {y[train_win:].std():.4f}")
print(f"R^2 OOS (Ridge on HAR features): {1 - residuals.var() / y[train_win:].var():.4f}")

In [ ]:
# Sanity peek: plot the residual series and overlay known crisis windows.
fig, ax = plt.subplots(figsize=(13, 2.6))
ax.plot(dates_resid, residuals, lw=0.3, c="k", alpha=0.6)
ax.axhline(0, c="gray", lw=0.5)
for label, lo, hi, color in [
    ("GFC", "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt", "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue", "2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon", "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID", "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23", "2023-03-08", "2023-03-22", "tab:green"),
]:
    ax.axvspan(pd.Timestamp(lo), pd.Timestamp(hi), alpha=0.18, color=color, label=label)
ax.legend(fontsize=7, ncol=6, loc="upper right")
ax.set_title("Ridge-OOS residuals over time, with marked crisis windows")
ax.set_ylim(np.quantile(residuals, [0.001, 0.999]))
plt.tight_layout(); plt.show()

## 3. The spectral embedding machinery

Cells below are marked `# export` — they collectively define
`build_embedding(views, d, k_graph, seed) -> SpectralBasis` plus the
sub-pieces it composes. Edit them here; `src/features/extractors/spectral_embedding.py`
is regenerated by hpc-agent's auto-export.

In [ ]:
# export
"""Temporal spectral embedding via graph Laplacian eigenmaps.

Pure feature transform: takes a stack of temporal views and returns a
low-dimensional coordinate per view. "Spectral" here refers to the
eigenspectrum of the graph Laplacian, not an FFT of the time series.
"""

from __future__ import annotations

from dataclasses import dataclass

import numpy as np
from scipy.sparse import csr_matrix, diags
from scipy.sparse.linalg import eigsh
from sklearn.neighbors import NearestNeighbors

In [ ]:
# export
def build_knn_graph(
    views: np.ndarray, k_graph: int
) -> tuple[csr_matrix, float, NearestNeighbors]:
    """Build a sparse symmetric k-NN affinity graph with self-tuning bandwidth.

    Each row of ``views`` is a point in R^W. Connects every point to its
    ``k_graph`` nearest neighbours (Euclidean), Gaussian edge weights
    ``w_ij = exp(-||v_i - v_j||^2 / (2 sigma^2))``. The bandwidth ``sigma``
    is the median ``k_graph``-th NN distance (Zelnik-Manor & Perona's
    self-tuning heuristic in its global-scale form).

    Returns
    -------
    W : symmetric Gaussian-weighted affinity matrix, shape (N, N).
    sigma : the bandwidth used.
    nn_index : fitted NearestNeighbors over ``views``; reusable for Nystrom.
    """
    n = views.shape[0]
    nn_index = NearestNeighbors(n_neighbors=k_graph + 1).fit(views)
    dists, idxs = nn_index.kneighbors(views)  # (N, k_graph+1); col 0 is self

    sigma = float(np.median(dists[:, k_graph]))

    rows = np.repeat(np.arange(n), k_graph)
    cols = idxs[:, 1:].ravel()
    vals = np.exp(-dists[:, 1:].ravel() ** 2 / (2 * sigma**2))
    graph = csr_matrix((vals, (rows, cols)), shape=(n, n))
    graph = graph.maximum(graph.T)  # symmetrize
    return graph, sigma, nn_index

In [ ]:
# export
def laplacian_eigenmaps(
    affinity: csr_matrix, d: int, seed: int = 42
) -> tuple[np.ndarray, np.ndarray]:
    """Bottom-d nontrivial eigenvectors of the symmetric normalized Laplacian.

    Computed as the top-d eigenvectors of M = D^{-1/2} W D^{-1/2} = I - L
    (eigsh with which='LA' is much faster than which='SM'). Same
    eigenvectors; eigenvalues map as ``lambda_L = 1 - lambda_M``.

    The leading eigenvalue is trivial (constant eigenvector proportional
    to ``sqrt(degree)``) and is dropped.

    Returns
    -------
    phi : (N, d) embedding coordinates.
    eigvals_L : (d,) Laplacian eigenvalues (ascending). Near-zero values
        beyond the first indicate the graph has multiple connected
        components.
    """
    n = affinity.shape[0]
    deg = np.asarray(affinity.sum(axis=1)).ravel()
    deg_safe = np.maximum(deg, 1e-12)
    d_inv_sqrt = diags(1.0 / np.sqrt(deg_safe))

    m = d_inv_sqrt @ affinity @ d_inv_sqrt  # m = I - L_sym

    rng = np.random.default_rng(seed)
    v0 = rng.standard_normal(n)
    eigvals_m, eigvecs = eigsh(m, k=d + 1, which="LA", v0=v0)

    order = np.argsort(-eigvals_m)  # descending in M <-> ascending in L
    eigvals_m = eigvals_m[order]
    eigvecs = eigvecs[:, order]

    phi = eigvecs[:, 1:]  # drop trivial top eigvec of M
    eigvals_l = 1.0 - eigvals_m[1:]
    return phi, eigvals_l

In [ ]:
# export
def nystrom_extend(
    v_test: np.ndarray,
    views_train: np.ndarray,
    phi_train: np.ndarray,
    sigma: float,
    k_graph: int,
    nn_index: NearestNeighbors | None = None,
) -> np.ndarray:
    """Extend a single test view to the embedding space (Bengio et al. 2003).

    Find the ``k_graph`` nearest training views, apply the same Gaussian
    kernel to get edge weights, return the kernel-weighted average of their
    training embeddings. Falls back to a uniform average if every neighbour
    has zero weight (extreme out-of-distribution test view).
    """
    if nn_index is None:
        nn_index = NearestNeighbors(n_neighbors=k_graph).fit(views_train)
    dists, idx = nn_index.kneighbors(v_test[None, :], n_neighbors=k_graph)
    dists = dists.ravel()
    idx = idx.ravel()

    weights = np.exp(-(dists**2) / (2 * sigma**2))
    weight_sum = weights.sum()
    if weight_sum <= 0:
        return phi_train[idx].mean(axis=0)
    weights = weights / weight_sum
    return (weights[:, None] * phi_train[idx]).sum(axis=0)

In [ ]:
# export
@dataclass
class SpectralBasis:
    """Frozen training-side embedding state — cheap to extend new views against."""

    views_train: np.ndarray  # (N, W)
    phi_train: np.ndarray  # (N, d)
    eigvals_L: np.ndarray  # (d,)
    sigma: float
    k_graph: int
    nn_index: NearestNeighbors

    def embed(self, v_test: np.ndarray) -> np.ndarray:
        """Embed a single test view via Nystrom. Returns shape (d,)."""
        return nystrom_extend(
            v_test,
            self.views_train,
            self.phi_train,
            self.sigma,
            self.k_graph,
            nn_index=self.nn_index,
        )

    def embed_batch(self, V_test: np.ndarray) -> np.ndarray:
        """Embed a batch of test views. Returns shape (M, d). Vectorized."""
        dists, idx = self.nn_index.kneighbors(V_test, n_neighbors=self.k_graph)
        weights = np.exp(-(dists**2) / (2 * self.sigma**2))
        weights = weights / np.clip(weights.sum(axis=1, keepdims=True), 1e-12, None)
        return np.einsum("mk,mkd->md", weights, self.phi_train[idx])


def build_embedding(
    views: np.ndarray, d: int, k_graph: int, seed: int = 42
) -> SpectralBasis:
    """End-to-end pipeline: views -> sparse k-NN graph -> Laplacian eigenmaps."""
    affinity, sigma, nn_index = build_knn_graph(views, k_graph=k_graph)
    phi, eigvals = laplacian_eigenmaps(affinity, d=d, seed=seed)
    return SpectralBasis(
        views_train=views,
        phi_train=phi,
        eigvals_L=eigvals,
        sigma=sigma,
        k_graph=k_graph,
        nn_index=nn_index,
    )

## 4. Form views + build the embedding

Each view is `W=960` consecutive residuals — 20 trading days. We **subsample to one view
per day** (every 48 bars) so the eigendecomp on the resulting graph is fast (~few seconds)
and the resulting scatter has ~4-5k points covering 2007-2024.

`build_embedding` is defined in the machinery section above (cells marked `# export`).

In [ ]:
VIEW_WINDOW = 960               # 20 days * 48 bars
SUBSAMPLE_STEP = PERIODS_PER_DAY  # one view per day
EMBEDDING_DIM = 8
GRAPH_K = 10

view_idx = np.arange(VIEW_WINDOW, len(residuals), SUBSAMPLE_STEP)
views = np.stack([residuals[i - VIEW_WINDOW : i] for i in view_idx])
view_dates = dates_resid.iloc[view_idx].reset_index(drop=True)

print(f"views shape: {views.shape}   (N views, W bars each)")
print(f"date span:   {view_dates.iloc[0]}  ..  {view_dates.iloc[-1]}")

In [ ]:
%time basis = build_embedding(views, d=EMBEDDING_DIM, k_graph=GRAPH_K, seed=42)

phi = basis.phi_train
print(f"phi: {phi.shape}    sigma: {basis.sigma:.4f}")
print(f"eigvals_L (small -> smoother coord): {basis.eigvals_L}")

In [ ]:
# Eigenvalue spectrum — a clear gap between the bottom-k and the rest suggests k useful dims.
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(range(1, EMBEDDING_DIM + 1), basis.eigvals_L, "o-")
ax.set_xlabel("eigenvector index")
ax.set_ylabel("Laplacian eigenvalue")
ax.set_title("Eigenvalue spectrum (lower = smoother / more informative)")
ax.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 5. What does the embedding actually see?

Three colorings of the same 2D scatter `phi_1` vs `phi_2`:

* **By year** — does the embedding cluster epochs? Does the GFC look different from
  the 2010s low-vol era?
* **By view RMS** — vol level proxy. If the embedding sorts by amplitude, the colors
  will gradient. If it sorts by *shape*, points of similar amplitude can be far apart.
* **By crisis-window membership** — overlay the same 6 named windows from the residual plot.
  If the embedding is doing something useful, crisis-window points cluster together.

In [ ]:
view_years = view_dates.dt.year.to_numpy()
view_rms = np.sqrt((views**2).mean(axis=1))

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

sc0 = axes[0].scatter(phi[:, 0], phi[:, 1], c=view_years, s=8, alpha=0.7, cmap="viridis")
axes[0].set_xlabel("phi_1"); axes[0].set_ylabel("phi_2")
axes[0].set_title("colored by year")
plt.colorbar(sc0, ax=axes[0], label="year")

sc1 = axes[1].scatter(phi[:, 0], phi[:, 1], c=view_rms, s=8, alpha=0.7,
                       cmap="plasma", norm=LogNorm())
axes[1].set_xlabel("phi_1"); axes[1].set_ylabel("phi_2")
axes[1].set_title("colored by view RMS (vol amplitude proxy, log)")
plt.colorbar(sc1, ax=axes[1], label="view RMS")

plt.tight_layout(); plt.show()

In [ ]:
# Highlight named crisis windows on top of the embedding.
WINDOWS = [
    ("GFC",          "2008-09-01", "2009-04-01", "tab:red"),
    ("EU debt",      "2011-07-01", "2011-10-30", "tab:orange"),
    ("China devalue","2015-08-15", "2015-10-01", "tab:purple"),
    ("Volmageddon",  "2018-02-01", "2018-02-20", "tab:brown"),
    ("COVID",        "2020-02-20", "2020-04-30", "tab:blue"),
    ("SVB / Mar23",  "2023-03-08", "2023-03-22", "tab:green"),
]

fig, ax = plt.subplots(figsize=(8, 6))
ax.scatter(phi[:, 0], phi[:, 1], c="lightgray", s=6, alpha=0.5, label="all views")
for label, lo, hi, color in WINDOWS:
    mask = (view_dates >= pd.Timestamp(lo)) & (view_dates < pd.Timestamp(hi))
    if mask.sum() == 0:
        continue
    ax.scatter(phi[mask, 0], phi[mask, 1], c=color, s=24, alpha=0.95,
               edgecolors="k", linewidths=0.4, label=f"{label}  (n={mask.sum()})")
ax.set_xlabel("phi_1"); ax.set_ylabel("phi_2")
ax.set_title("Embedding with named crisis windows highlighted")
ax.legend(fontsize=8, loc="best")
plt.tight_layout(); plt.show()

## 6. Higher dims — does anything beyond phi_1, phi_2 matter?

Pairs plot of the first 4 dims. If phi_3 / phi_4 just look like noise around zero, the
useful dimensionality is 2-3. If they show structure (clusters, arcs), `d` should stay >= 4.

In [ ]:
fig, axes = plt.subplots(3, 3, figsize=(11, 11))
for i in range(3):
    for j in range(3):
        ax = axes[i, j]
        if i <= j:
            ax.set_visible(False)
            continue
        ax.scatter(phi[:, j], phi[:, i + 1], c=view_years, cmap="viridis", s=4, alpha=0.6)
        ax.set_xlabel(f"phi_{j + 1}")
        ax.set_ylabel(f"phi_{i + 2}")
plt.suptitle("First 4 embedding dims, pairs colored by year", y=1.02)
plt.tight_layout(); plt.show()

## 7. Nystrom out-of-sample extension

The full pipeline embeds **new** views (one per backtest step) into the frozen training basis.
Sanity-check that machinery: take the last 200 views, treat them as "new," Nyström-extend them,
compare to their actual training embeddings.

(Slightly cheats — those points were in the training set. The point is just to confirm Nyström
doesn't catastrophically mis-place known points.)

In [ ]:
n_check = 200
check_idx = np.arange(len(views) - n_check, len(views))
phi_recovered = basis.embed_batch(views[check_idx])
rmse = np.sqrt(((phi_recovered - phi[check_idx]) ** 2).mean(axis=1))
phi_scale = phi.std(axis=0).mean()

print(f"Nystrom-vs-true RMSE on last 200 training views:")
print(f"  median:   {np.median(rmse):.4f}")
print(f"  max:      {rmse.max():.4f}")
print(f"  vs embedding scale (mean std per dim): {phi_scale:.4f}")
print(f"  relative median:  {np.median(rmse) / phi_scale * 100:.1f}% of embedding scale")

## What to look for / what tells you it's working

* **Crisis-window highlights cluster.** If the GFC / COVID dots land in a tight blob distinct
  from quiet-market dots, the embedding is finding regime structure. If they scatter randomly,
  the embedding is mostly noise.
* **Year colormap shows arc / drift.** A smooth color gradient suggests the embedding tracks
  slow regime drift over years. Discontinuities suggest abrupt regime shifts.
* **RMS coloring shows non-trivial layout.** If `phi` were just a rotation of view amplitude,
  the RMS scatter would be a clean radial gradient. Anything more interesting (e.g. two clusters
  at the same amplitude) means the embedding separates *shapes* of residual trajectories,
  not just amplitudes — that's the whole point.
* **Eigenvalue gap.** A visible gap between the first few eigenvalues and the rest is the
  classical signal of "there are k natural clusters / dimensions here."
* **Nystrom RMSE < 10% of embedding scale.** Confirms the out-of-sample extension used in
  the spectral_knn pipeline at test time isn't doing anything pathological.